# 05.10_CAFE5_node_GO_KEGG_R

指定节点 GO/KEGG 注释与富集。

- 当前文件：`analysis/05_genome_analysis/05.10_CAFE5_node_GO_KEGG_R.ipynb`
- 原始来源：`Codes/05.10_R_CAFE5_node_GO_KEGG.ipynb`（旧编号仅用于溯源）。
- 运行内核：**R**。
- 导入依赖：`clusterProfiler`, `dplyr`, `ggplot2`, `ggrepel`, `stringr`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。

**本文件说明：** 含手动去除注释表 query 前 # 的记录；05.11 富集分支还读取 06 阶段建立的 TERM2GENE/TERM2NAME 等参考表，编号不是严格拓扑顺序。


## 1. GO注释
详见/share/home/zhangze/zz/NeuralOrigin/Codes/05.08_get_GO_basic.sh

## 2. KEGG注释
详见/share/home/zhangze/zz/NeuralOrigin/Codes/05.09_R_get_KEGG.ipynb

## 3. Node11

### 3.1 背景基因注释构建

手动将node11.emapper.annotations中的query前的井号#去掉

In [ ]:
library (clusterProfiler)
library (dplyr)
library (stringr)
options (stringsAsFactors = F)

In [ ]:
##===============================STEP1:GO 注释生成 =======================
# 自己构建的话，首先需要读入文件
# egg <- read.delim ("node25.emapper.annotations",header = T,sep="\t")
egg <- read.delim ("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node11.emapper.annotations",header = T,sep="\t")
egg [egg==""]<-NA  #将空行变成 NA，方便下面的去除
# 从文件中挑出基因 query 与 eggnog 注释信息
##gene_info <- egg %>% 
##  dplyr::select (GID = query, GENENAME = eggNOG_OGs) %>% na.omit ()
# 挑出 query_name 与 GO 注释信息
gterms <- egg %>%
  dplyr::select (query, GOs) %>% na.omit ()
gene_ids <- egg$query
eggnog_lines_with_go <- egg$GOs!= ""
eggnog_lines_with_go
eggnog_annoations_go <- str_split (egg [eggnog_lines_with_go,]$GOs, ",")
gene2go <- data.frame (gene = rep (gene_ids [eggnog_lines_with_go],
                                    times = sapply (eggnog_annoations_go, length)),
                         term = unlist (eggnog_annoations_go))
names (gene2go) <- c ('gene_id', 'ID')
# go2name <- read.delim ('GO.library', header = FALSE, stringsAsFactors = FALSE)
go2name <- read.delim ('/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/GO_and_KEGG/GO.library', header = FALSE, stringsAsFactors = FALSE)
names (go2name) <- c ('ID', 'Description', 'Ontology')

go_anno <- merge (gene2go, go2name, by = 'ID', all.x = TRUE)

## 将 GO 注释信息保存
save (go_anno,file = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node11_GO.rda")

In [ ]:
##===============================STEP2:KEGG 注释生成 =======================
load("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/GO_and_KEGG/kegg_info.RData")

gene2ko <- egg %>%
  dplyr::select (GID = query, KO = KEGG_ko) %>%
  na.omit ()
pathway2name <- read.delim ("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/GO_and_KEGG/KEGG.library")
colnames (pathway2name)<-c ("Pathway","Name")
gene2ko$KO <- str_replace (gene2ko$KO, "ko:","")
gene2pathway <- gene2ko %>% left_join (ko2pathway, by = "KO") %>% 
  dplyr::select (GID, Pathway) %>%
  na.omit ()
kegg_anno<- merge (gene2pathway,pathway2name,by = 'Pathway', all.x = TRUE)[,c (2,1,3)]
colnames (kegg_anno) <- c ('gene_id','pathway_id','pathway_description')
save (kegg_anno,file = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node11_KEGG.rda")

### 3.2 富集分析

In [ ]:
##===============================STEP3:GO 富集分析 =================
# 目标基因列表 (全部基因)
# gene_select <- read.delim (file = 'node25significant.expand.genes', stringsAsFactors = FALSE,header = F)$V1
gene_select <- read.delim (file = '/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node11significant.expand.genes.txt', stringsAsFactors = FALSE,header = F)$V1
#GO 富集分析
# 默认以所有注释到 GO 的基因为背景集，也可通过 universe 参数输入背景集
# 默认以 p<0.05 为标准，Benjamini 方法校正 p 值，q 值阈值 0.2
# 默认输出 top500 富集结果
# 如果想输出所有富集结果（不考虑 p 值阈值等），将 p、q 等值设置为 1 即可
# 或者直接在 enrichResult 类对象中直接提取需要的结果
go_rich <- enricher (gene = gene_select,
                    TERM2GENE = go_anno [c ('ID', 'gene_id')], 
                    TERM2NAME = go_anno [c ('ID', 'Description')], 
                    pvalueCutoff = 1, 
                    pAdjustMethod = 'BH', 
                    qvalueCutoff = 1
                   )
# 输出默认结果，即根据上述 p 值等阈值筛选后的


# tmp <- merge (go_rich, go2name [c ('ID', 'Ontology')], by = 'ID')
# tmp <- tmp [c (10, 1:9)]
# tmp <- tmp [order (tmp$pvalue), ]
# write.table (tmp, '/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/CAFE_output/node11significant.expand.GO.xls', sep = '\t', row.names = FALSE, quote = FALSE)
# 正确保存所有富集条目的详细信息
go_rich_res <- as.data.frame(go_rich)
write.table(go_rich_res, '/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node11significant.expand.GO.xls', sep = '\t', row.names = FALSE, quote = FALSE)

In [ ]:
##===============================STEP4:KEGG 注释 =================
# gene_select <- read.delim ('node25significant.expand.genes', stringsAsFactors = FALSE,header = F)$V1
gene_select <- read.delim ('/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node11significant.expand.genes.txt', stringsAsFactors = FALSE,header = F)$V1
#KEGG 富集分析
# 默认以所有注释到 KEGG 的基因为背景集，也可通过 universe 参数指定其中的一个子集作为背景集
# 默认以 p<0.05 为标准，Benjamini 方法校正 p 值，q 值阈值 0.2
# 默认输出 top500 富集结果
kegg_rich <- enricher (gene = gene_select,
                      TERM2GENE = kegg_anno [c ('pathway_id', 'gene_id')], 
                      TERM2NAME = kegg_anno [c ('pathway_id', 'pathway_description')], 
                      pvalueCutoff = 1, 
                      pAdjustMethod = 'BH', 
                      qvalueCutoff = 1, 
                      maxGSSize = 500)

# # 输出默认结果，即根据上述 p 值等阈值筛选后的
# write.table (kegg_rich, '/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/CAFE_output/node11significant.expand.KEGG.xls', sep = '\t', row.names = FALSE, quote = FALSE)
kegg_rich_res <- as.data.frame(kegg_rich)
write.table(kegg_rich_res, '/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node11significant.expand.KEGG.xls', sep = '\t', row.names = FALSE, quote = FALSE)

### 3.3 绘图

> 需要注意生成的文件中有一些 term 可能是重复的，比如 GO 富集结果的 16,17 行，仔细看 geneID 那一列就会发现两行的基因是完全一样的，需要删掉一个。还有一点就是在 GO/KEGG 富集的结果中，有一些是与所研究对象物种无关的 ，这些也请酌情考虑

In [ ]:
# 读入原始GO富集文件
df <- read.delim("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node11significant.expand.GO.xls", header = TRUE, sep = "\t", stringsAsFactors = FALSE)

# 去除geneID完全重复的行
# df_processed <- df[!duplicated(df$geneID), ]

# 去除Description重复，保留第一个（即p值最小的那个）
df_processed <- df[!duplicated(df$Description), ]

# 保存结果
write.table(df_processed, "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node11significant.expand.GO.processed.xls", sep = "\t", row.names = FALSE, quote = FALSE)

In [ ]:
library(ggplot2)
library(dplyr)
library(ggrepel)

# 读入数据
pathway <- read.delim(
  "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node11significant.expand.GO.processed.xls",
  header = TRUE, sep = "\t"
)

# GeneRatio直接用Count列
pathway$GeneRatio <- pathway$Count / 16091

# logP
pathway$logP <- -log10(pathway$pvalue)

# 生成唯一GO标签，防止Description重复造成y轴合并
pathway$GO_label <- paste0(pathway$Description, " | ", pathway$ID)

# 按pvalue或p.adjust排序，选前20
GO <- arrange(pathway, pvalue)   # 或者 p.adjust
GO_dataset <- GO[1:20, ]
GO_dataset$GO_label <- factor(GO_dataset$GO_label, levels = rev(GO_dataset$GO_label))

# Ontology如无可区分，统一赋值
GO_dataset$Ontology <- "BP"

# 绘图主题
mytheme <- theme(
  axis.title = element_text(face="bold", size=8, colour='black'),
  axis.text.y = element_text(face="bold", size=6, colour='black'),
  axis.text.x = element_text(face="bold", color="black", angle=0, vjust=1, size=8),
  axis.line = element_line(linewidth=0.5, colour = 'black'),
  panel.background = element_rect(color='black'),
  plot.title = element_text(face="bold", size=8, colour = 'black', hjust = 0.8),
  legend.key = element_blank()
)

# 气泡图（y轴唯一标识）
p <- ggplot(GO_dataset, aes(x=GeneRatio, y=GO_label, colour=logP, size=Count, shape=Ontology)) +
  geom_point() +
  scale_size(range = c(2, 8)) +
  scale_colour_gradient(low = "#52c2eb", high = "#EA4F30") +
  theme_bw() +
  labs(
    x='GeneRatio',
    y='GO Terms (Description | ID)',
    title='Enriched GO Terms',
    color=expression(-log[10](pvalue))
  ) +
  theme(
    legend.title = element_text(size=8),
    legend.text = element_text(size=14),
    axis.title.y = element_text(margin = margin(r = 50)),
    axis.title.x = element_text(margin = margin(t = 20)),
    axis.text.x = element_text(face="bold", color="black", angle=0, vjust=1)
  ) +
  mytheme

print(p)

# 保存图片
ggsave(p, filename = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node11GO.pdf", width = 210, height = 210, units = "mm", dpi=300)
ggsave(p, filename = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node11GO.png", width = 210, height = 210, units = "mm", dpi=300)


## 4. Node12

### 4.1 背景基因注释构建

手动将node12.emapper.annotations中的query前的井号#去掉

In [ ]:
library (clusterProfiler)
library (dplyr)
library (stringr)
options (stringsAsFactors = F)

In [ ]:
##===============================STEP1:GO 注释生成 =======================
# 自己构建的话，首先需要读入文件
# egg <- read.delim ("node25.emapper.annotations",header = T,sep="\t")
egg <- read.delim ("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node12.emapper.annotations",header = T,sep="\t")
egg [egg==""]<-NA  #将空行变成 NA，方便下面的去除
# 从文件中挑出基因 query 与 eggnog 注释信息
##gene_info <- egg %>% 
##  dplyr::select (GID = query, GENENAME = eggNOG_OGs) %>% na.omit ()
# 挑出 query_name 与 GO 注释信息
gterms <- egg %>%
  dplyr::select (query, GOs) %>% na.omit ()
gene_ids <- egg$query
eggnog_lines_with_go <- egg$GOs!= ""
eggnog_lines_with_go
eggnog_annoations_go <- str_split (egg [eggnog_lines_with_go,]$GOs, ",")
gene2go <- data.frame (gene = rep (gene_ids [eggnog_lines_with_go],
                                    times = sapply (eggnog_annoations_go, length)),
                         term = unlist (eggnog_annoations_go))
names (gene2go) <- c ('gene_id', 'ID')
# go2name <- read.delim ('GO.library', header = FALSE, stringsAsFactors = FALSE)
go2name <- read.delim ('/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/GO_and_KEGG/GO.library', header = FALSE, stringsAsFactors = FALSE)
names (go2name) <- c ('ID', 'Description', 'Ontology')

go_anno <- merge (gene2go, go2name, by = 'ID', all.x = TRUE)

## 将 GO 注释信息保存
save (go_anno,file = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node12_GO.rda")

In [ ]:
##===============================STEP2:KEGG 注释生成 =======================
load("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/GO_and_KEGG/kegg_info.RData")

gene2ko <- egg %>%
  dplyr::select (GID = query, KO = KEGG_ko) %>%
  na.omit ()
pathway2name <- read.delim ("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/GO_and_KEGG/KEGG.library")
colnames (pathway2name)<-c ("Pathway","Name")
gene2ko$KO <- str_replace (gene2ko$KO, "ko:","")
gene2pathway <- gene2ko %>% left_join (ko2pathway, by = "KO") %>% 
  dplyr::select (GID, Pathway) %>%
  na.omit ()
kegg_anno<- merge (gene2pathway,pathway2name,by = 'Pathway', all.x = TRUE)[,c (2,1,3)]
colnames (kegg_anno) <- c ('gene_id','pathway_id','pathway_description')
save (kegg_anno,file = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node12_KEGG.rda")

### 4.2 富集分析

In [ ]:
##===============================STEP3:GO 富集分析 =================
# 目标基因列表 (全部基因)
# gene_select <- read.delim (file = 'node25significant.expand.genes', stringsAsFactors = FALSE,header = F)$V1
gene_select <- read.delim (file = '/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node12significant.expand.genes.txt', stringsAsFactors = FALSE,header = F)$V1
#GO 富集分析
# 默认以所有注释到 GO 的基因为背景集，也可通过 universe 参数输入背景集
# 默认以 p<0.05 为标准，Benjamini 方法校正 p 值，q 值阈值 0.2
# 默认输出 top500 富集结果
# 如果想输出所有富集结果（不考虑 p 值阈值等），将 p、q 等值设置为 1 即可
# 或者直接在 enrichResult 类对象中直接提取需要的结果
go_rich <- enricher (gene = gene_select,
                    TERM2GENE = go_anno [c ('ID', 'gene_id')], 
                    TERM2NAME = go_anno [c ('ID', 'Description')], 
                    pvalueCutoff = 1, 
                    pAdjustMethod = 'BH', 
                    qvalueCutoff = 1
                   )
# 输出默认结果，即根据上述 p 值等阈值筛选后的


# tmp <- merge (go_rich, go2name [c ('ID', 'Ontology')], by = 'ID')
# tmp <- tmp [c (10, 1:9)]
# tmp <- tmp [order (tmp$pvalue), ]
# write.table (tmp, '/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/CAFE_output/node12significant.expand.GO.xls', sep = '\t', row.names = FALSE, quote = FALSE)
# 正确保存所有富集条目的详细信息
go_rich_res <- as.data.frame(go_rich)
write.table(go_rich_res, '/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node12significant.expand.GO.xls', sep = '\t', row.names = FALSE, quote = FALSE)

In [ ]:
##===============================STEP4:KEGG 注释 =================
# gene_select <- read.delim ('node25significant.expand.genes', stringsAsFactors = FALSE,header = F)$V1
gene_select <- read.delim ('/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node12significant.expand.genes.txt', stringsAsFactors = FALSE,header = F)$V1
#KEGG 富集分析
# 默认以所有注释到 KEGG 的基因为背景集，也可通过 universe 参数指定其中的一个子集作为背景集
# 默认以 p<0.05 为标准，Benjamini 方法校正 p 值，q 值阈值 0.2
# 默认输出 top500 富集结果
kegg_rich <- enricher (gene = gene_select,
                      TERM2GENE = kegg_anno [c ('pathway_id', 'gene_id')], 
                      TERM2NAME = kegg_anno [c ('pathway_id', 'pathway_description')], 
                      pvalueCutoff = 1, 
                      pAdjustMethod = 'BH', 
                      qvalueCutoff = 1, 
                      maxGSSize = 500)

# # 输出默认结果，即根据上述 p 值等阈值筛选后的
# write.table (kegg_rich, '/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/CAFE_output/node12significant.expand.KEGG.xls', sep = '\t', row.names = FALSE, quote = FALSE)
kegg_rich_res <- as.data.frame(kegg_rich)
write.table(kegg_rich_res, '/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node12significant.expand.KEGG.xls', sep = '\t', row.names = FALSE, quote = FALSE)

### 4.3 绘图

> 需要注意生成的文件中有一些 term 可能是重复的，比如 GO 富集结果的 16,17 行，仔细看 geneID 那一列就会发现两行的基因是完全一样的，需要删掉一个。还有一点就是在 GO/KEGG 富集的结果中，有一些是与所研究对象物种无关的 ，这些也请酌情考虑

In [ ]:
# 读入原始GO富集文件
df <- read.delim("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node12significant.expand.GO.xls", header = TRUE, sep = "\t", stringsAsFactors = FALSE)

# 去除geneID完全重复的行
# df_processed <- df[!duplicated(df$geneID), ]

# 去除Description重复，保留第一个（即p值最小的那个）
df_processed <- df[!duplicated(df$Description), ]

# 保存结果
write.table(df_processed, "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node12significant.expand.GO.processed.xls", sep = "\t", row.names = FALSE, quote = FALSE)

In [ ]:
library(ggplot2)
library(dplyr)
library(ggrepel)

# 读入数据
pathway <- read.delim(
  "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node12significant.expand.GO.processed.xls",
  header = TRUE, sep = "\t"
)

# GeneRatio直接用Count列
pathway$GeneRatio <- pathway$Count / 9742

# logP
pathway$logP <- -log10(pathway$pvalue)

# 生成唯一GO标签，防止Description重复造成y轴合并
pathway$GO_label <- paste0(pathway$Description, " | ", pathway$ID)

# 按pvalue或p.adjust排序，选前20
GO <- arrange(pathway, pvalue)   # 或者 p.adjust
GO_dataset <- GO[1:20, ]
GO_dataset$GO_label <- factor(GO_dataset$GO_label, levels = rev(GO_dataset$GO_label))

# Ontology如无可区分，统一赋值
GO_dataset$Ontology <- "BP"

# 绘图主题
mytheme <- theme(
  axis.title = element_text(face="bold", size=8, colour='black'),
  axis.text.y = element_text(face="bold", size=6, colour='black'),
  axis.text.x = element_text(face="bold", color="black", angle=0, vjust=1, size=8),
  axis.line = element_line(linewidth=0.5, colour = 'black'),
  panel.background = element_rect(color='black'),
  plot.title = element_text(face="bold", size=8, colour = 'black', hjust = 0.8),
  legend.key = element_blank()
)

# 气泡图（y轴唯一标识）
p <- ggplot(GO_dataset, aes(x=GeneRatio, y=GO_label, colour=logP, size=Count, shape=Ontology)) +
  geom_point() +
  scale_size(range = c(2, 8)) +
  scale_colour_gradient(low = "#52c2eb", high = "#EA4F30") +
  theme_bw() +
  labs(
    x='GeneRatio',
    y='GO Terms (Description | ID)',
    title='Enriched GO Terms',
    color=expression(-log[10](pvalue))
  ) +
  theme(
    legend.title = element_text(size=8),
    legend.text = element_text(size=14),
    axis.title.y = element_text(margin = margin(r = 50)),
    axis.title.x = element_text(margin = margin(t = 20)),
    axis.text.x = element_text(face="bold", color="black", angle=0, vjust=1)
  ) +
  mytheme

print(p)

# 保存图片
ggsave(p, filename = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node12GO.pdf", width = 210, height = 210, units = "mm", dpi=300)
ggsave(p, filename = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node12GO.png", width = 210, height = 210, units = "mm", dpi=300)


## 5. Node15

### 5.1 背景基因注释构建

手动将node15.emapper.annotations中的query前的井号#去掉

In [ ]:
library (clusterProfiler)
library (dplyr)
library (stringr)
options (stringsAsFactors = F)

In [ ]:
##===============================STEP1:GO 注释生成 =======================
# 自己构建的话，首先需要读入文件
# egg <- read.delim ("node25.emapper.annotations",header = T,sep="\t")
egg <- read.delim ("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node15.emapper.annotations",header = T,sep="\t")
egg [egg==""]<-NA  #将空行变成 NA，方便下面的去除
# 从文件中挑出基因 query 与 eggnog 注释信息
##gene_info <- egg %>% 
##  dplyr::select (GID = query, GENENAME = eggNOG_OGs) %>% na.omit ()
# 挑出 query_name 与 GO 注释信息
gterms <- egg %>%
  dplyr::select (query, GOs) %>% na.omit ()
gene_ids <- egg$query
eggnog_lines_with_go <- egg$GOs!= ""
eggnog_lines_with_go
eggnog_annoations_go <- str_split (egg [eggnog_lines_with_go,]$GOs, ",")
gene2go <- data.frame (gene = rep (gene_ids [eggnog_lines_with_go],
                                    times = sapply (eggnog_annoations_go, length)),
                         term = unlist (eggnog_annoations_go))
names (gene2go) <- c ('gene_id', 'ID')
# go2name <- read.delim ('GO.library', header = FALSE, stringsAsFactors = FALSE)
go2name <- read.delim ('/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/GO_and_KEGG/GO.library', header = FALSE, stringsAsFactors = FALSE)
names (go2name) <- c ('ID', 'Description', 'Ontology')

go_anno <- merge (gene2go, go2name, by = 'ID', all.x = TRUE)

## 将 GO 注释信息保存
save (go_anno,file = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node15_GO.rda")

In [ ]:
##===============================STEP2:KEGG 注释生成 =======================
load("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/GO_and_KEGG/kegg_info.RData")

gene2ko <- egg %>%
  dplyr::select (GID = query, KO = KEGG_ko) %>%
  na.omit ()
pathway2name <- read.delim ("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/GO_and_KEGG/KEGG.library")
colnames (pathway2name)<-c ("Pathway","Name")
gene2ko$KO <- str_replace (gene2ko$KO, "ko:","")
gene2pathway <- gene2ko %>% left_join (ko2pathway, by = "KO") %>% 
  dplyr::select (GID, Pathway) %>%
  na.omit ()
kegg_anno<- merge (gene2pathway,pathway2name,by = 'Pathway', all.x = TRUE)[,c (2,1,3)]
colnames (kegg_anno) <- c ('gene_id','pathway_id','pathway_description')
save (kegg_anno,file = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node15_KEGG.rda")

### 5.2 富集分析

In [ ]:
##===============================STEP3:GO 富集分析 =================
# 目标基因列表 (全部基因)
# gene_select <- read.delim (file = 'node25significant.expand.genes', stringsAsFactors = FALSE,header = F)$V1
gene_select <- read.delim (file = '/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node15significant.contract.genes.txt', stringsAsFactors = FALSE,header = F)$V1
#GO 富集分析
# 默认以所有注释到 GO 的基因为背景集，也可通过 universe 参数输入背景集
# 默认以 p<0.05 为标准，Benjamini 方法校正 p 值，q 值阈值 0.2
# 默认输出 top500 富集结果
# 如果想输出所有富集结果（不考虑 p 值阈值等），将 p、q 等值设置为 1 即可
# 或者直接在 enrichResult 类对象中直接提取需要的结果
go_rich <- enricher (gene = gene_select,
                    TERM2GENE = go_anno [c ('ID', 'gene_id')], 
                    TERM2NAME = go_anno [c ('ID', 'Description')], 
                    pvalueCutoff = 1, 
                    pAdjustMethod = 'BH', 
                    qvalueCutoff = 1
                   )
# 输出默认结果，即根据上述 p 值等阈值筛选后的


# tmp <- merge (go_rich, go2name [c ('ID', 'Ontology')], by = 'ID')
# tmp <- tmp [c (10, 1:9)]
# tmp <- tmp [order (tmp$pvalue), ]
# write.table (tmp, '/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/CAFE_output/node12significant.expand.GO.xls', sep = '\t', row.names = FALSE, quote = FALSE)
# 正确保存所有富集条目的详细信息
go_rich_res <- as.data.frame(go_rich)
write.table(go_rich_res, '/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node15significant.contract.GO.xls', sep = '\t', row.names = FALSE, quote = FALSE)

In [ ]:
##===============================STEP4:KEGG 注释 =================
# gene_select <- read.delim ('node25significant.expand.genes', stringsAsFactors = FALSE,header = F)$V1
gene_select <- read.delim ('/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node15significant.contract.genes.txt', stringsAsFactors = FALSE,header = F)$V1
#KEGG 富集分析
# 默认以所有注释到 KEGG 的基因为背景集，也可通过 universe 参数指定其中的一个子集作为背景集
# 默认以 p<0.05 为标准，Benjamini 方法校正 p 值，q 值阈值 0.2
# 默认输出 top500 富集结果
kegg_rich <- enricher (gene = gene_select,
                      TERM2GENE = kegg_anno [c ('pathway_id', 'gene_id')], 
                      TERM2NAME = kegg_anno [c ('pathway_id', 'pathway_description')], 
                      pvalueCutoff = 1, 
                      pAdjustMethod = 'BH', 
                      qvalueCutoff = 1, 
                      maxGSSize = 500)

# # 输出默认结果，即根据上述 p 值等阈值筛选后的
# write.table (kegg_rich, '/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/CAFE_output/node12significant.expand.KEGG.xls', sep = '\t', row.names = FALSE, quote = FALSE)
kegg_rich_res <- as.data.frame(kegg_rich)
write.table(kegg_rich_res, '/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node15significant.contract.KEGG.xls', sep = '\t', row.names = FALSE, quote = FALSE)

### 5.3 绘图

> 需要注意生成的文件中有一些 term 可能是重复的，比如 GO 富集结果的 16,17 行，仔细看 geneID 那一列就会发现两行的基因是完全一样的，需要删掉一个。还有一点就是在 GO/KEGG 富集的结果中，有一些是与所研究对象物种无关的 ，这些也请酌情考虑

In [ ]:
# 读入原始GO富集文件
df <- read.delim("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node15significant.contract.GO.xls", header = TRUE, sep = "\t", stringsAsFactors = FALSE)

# 去除geneID完全重复的行
# df_processed <- df[!duplicated(df$geneID), ]

# 去除Description重复，保留第一个（即p值最小的那个）
df_processed <- df[!duplicated(df$Description), ]

# 保存结果
write.table(df_processed, "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node15significant.contract.GO.processed.xls", sep = "\t", row.names = FALSE, quote = FALSE)

In [ ]:
library(ggplot2)
library(dplyr)
library(ggrepel)

# 读入数据
pathway <- read.delim(
  "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node15significant.contract.GO.processed.xls",
  header = TRUE, sep = "\t"
)

# GeneRatio直接用Count列
pathway$GeneRatio <- pathway$Count / 19831

# logP
pathway$logP <- -log10(pathway$pvalue)

# 生成唯一GO标签，防止Description重复造成y轴合并
pathway$GO_label <- paste0(pathway$Description, " | ", pathway$ID)

# 按pvalue或p.adjust排序，选前20
GO <- arrange(pathway, pvalue)   # 或者 p.adjust
GO_dataset <- GO[1:20, ]
GO_dataset$GO_label <- factor(GO_dataset$GO_label, levels = rev(GO_dataset$GO_label))

# Ontology如无可区分，统一赋值
GO_dataset$Ontology <- "BP"

# 绘图主题
mytheme <- theme(
  axis.title = element_text(face="bold", size=8, colour='black'),
  axis.text.y = element_text(face="bold", size=6, colour='black'),
  axis.text.x = element_text(face="bold", color="black", angle=0, vjust=1, size=8),
  axis.line = element_line(linewidth=0.5, colour = 'black'),
  panel.background = element_rect(color='black'),
  plot.title = element_text(face="bold", size=8, colour = 'black', hjust = 0.8),
  legend.key = element_blank()
)

# 气泡图（y轴唯一标识）
p <- ggplot(GO_dataset, aes(x=GeneRatio, y=GO_label, colour=logP, size=Count, shape=Ontology)) +
  geom_point() +
  scale_size(range = c(2, 8)) +
  scale_colour_gradient(low = "#52c2eb", high = "#EA4F30") +
  theme_bw() +
  labs(
    x='GeneRatio',
    y='GO Terms (Description | ID)',
    title='Enriched GO Terms',
    color=expression(-log[10](pvalue))
  ) +
  theme(
    legend.title = element_text(size=8),
    legend.text = element_text(size=14),
    axis.title.y = element_text(margin = margin(r = 50)),
    axis.title.x = element_text(margin = margin(t = 20)),
    axis.text.x = element_text(face="bold", color="black", angle=0, vjust=1)
  ) +
  mytheme

print(p)

# 保存图片
ggsave(p, filename = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node15GO.pdf", width = 210, height = 210, units = "mm", dpi=300)
ggsave(p, filename = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node15GO.png", width = 210, height = 210, units = "mm", dpi=300)


## 6. Node12 + Node15

### 6.1 背景基因注释构建

手动将nodeCommon_12_15.emapper.annotations中的query前的井号#去掉

In [ ]:
library (clusterProfiler)
library (dplyr)
library (stringr)
options (stringsAsFactors = F)

In [ ]:
##===============================STEP1:GO 注释生成 =======================
# 自己构建的话，首先需要读入文件
# egg <- read.delim ("node25.emapper.annotations",header = T,sep="\t")
egg <- read.delim ("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/nodeCommon_12_15.emapper.annotations",header = T,sep="\t")
egg [egg==""]<-NA  #将空行变成 NA，方便下面的去除
# 从文件中挑出基因 query 与 eggnog 注释信息
##gene_info <- egg %>% 
##  dplyr::select (GID = query, GENENAME = eggNOG_OGs) %>% na.omit ()
# 挑出 query_name 与 GO 注释信息
gterms <- egg %>%
  dplyr::select (query, GOs) %>% na.omit ()
gene_ids <- egg$query
eggnog_lines_with_go <- egg$GOs!= ""
eggnog_lines_with_go
eggnog_annoations_go <- str_split (egg [eggnog_lines_with_go,]$GOs, ",")
gene2go <- data.frame (gene = rep (gene_ids [eggnog_lines_with_go],
                                    times = sapply (eggnog_annoations_go, length)),
                         term = unlist (eggnog_annoations_go))
names (gene2go) <- c ('gene_id', 'ID')
# go2name <- read.delim ('GO.library', header = FALSE, stringsAsFactors = FALSE)
go2name <- read.delim ('/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/GO_and_KEGG/GO.library', header = FALSE, stringsAsFactors = FALSE)
names (go2name) <- c ('ID', 'Description', 'Ontology')

go_anno <- merge (gene2go, go2name, by = 'ID', all.x = TRUE)

## 将 GO 注释信息保存
save (go_anno,file = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/nodeCommon_12_15_GO.rda")

In [ ]:
##===============================STEP2:KEGG 注释生成 =======================
load("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/GO_and_KEGG/kegg_info.RData")

gene2ko <- egg %>%
  dplyr::select (GID = query, KO = KEGG_ko) %>%
  na.omit ()
pathway2name <- read.delim ("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/GO_and_KEGG/KEGG.library")
colnames (pathway2name)<-c ("Pathway","Name")
gene2ko$KO <- str_replace (gene2ko$KO, "ko:","")
gene2pathway <- gene2ko %>% left_join (ko2pathway, by = "KO") %>% 
  dplyr::select (GID, Pathway) %>%
  na.omit ()
kegg_anno<- merge (gene2pathway,pathway2name,by = 'Pathway', all.x = TRUE)[,c (2,1,3)]
colnames (kegg_anno) <- c ('gene_id','pathway_id','pathway_description')
save (kegg_anno,file = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/nodeCommon_12_15_KEGG.rda")

### 6.2 富集分析

In [ ]:
##===============================STEP3:GO 富集分析 =================
# 目标基因列表 (全部基因)
# gene_select <- read.delim (file = 'node25significant.expand.genes', stringsAsFactors = FALSE,header = F)$V1
gene_select <- read.delim (file = '/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/nodeCommon_12_15_significant.intersect.genes.txt', stringsAsFactors = FALSE,header = F)$V1
#GO 富集分析
# 默认以所有注释到 GO 的基因为背景集，也可通过 universe 参数输入背景集
# 默认以 p<0.05 为标准，Benjamini 方法校正 p 值，q 值阈值 0.2
# 默认输出 top500 富集结果
# 如果想输出所有富集结果（不考虑 p 值阈值等），将 p、q 等值设置为 1 即可
# 或者直接在 enrichResult 类对象中直接提取需要的结果
go_rich <- enricher (gene = gene_select,
                    TERM2GENE = go_anno [c ('ID', 'gene_id')], 
                    TERM2NAME = go_anno [c ('ID', 'Description')], 
                    pvalueCutoff = 1, 
                    pAdjustMethod = 'BH', 
                    qvalueCutoff = 1
                   )
# 输出默认结果，即根据上述 p 值等阈值筛选后的


# tmp <- merge (go_rich, go2name [c ('ID', 'Ontology')], by = 'ID')
# tmp <- tmp [c (10, 1:9)]
# tmp <- tmp [order (tmp$pvalue), ]
# write.table (tmp, '/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/CAFE_output/node12significant.expand.GO.xls', sep = '\t', row.names = FALSE, quote = FALSE)
# 正确保存所有富集条目的详细信息
go_rich_res <- as.data.frame(go_rich)
write.table(go_rich_res, '/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/nodeCommon_12_15_significant.intersect.GO.xls', sep = '\t', row.names = FALSE, quote = FALSE)

In [ ]:
##===============================STEP4:KEGG 注释 =================
# gene_select <- read.delim ('node25significant.expand.genes', stringsAsFactors = FALSE,header = F)$V1
gene_select <- read.delim ('/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/nodeCommon_12_15_significant.intersect.genes.txt', stringsAsFactors = FALSE,header = F)$V1
#KEGG 富集分析
# 默认以所有注释到 KEGG 的基因为背景集，也可通过 universe 参数指定其中的一个子集作为背景集
# 默认以 p<0.05 为标准，Benjamini 方法校正 p 值，q 值阈值 0.2
# 默认输出 top500 富集结果
kegg_rich <- enricher (gene = gene_select,
                      TERM2GENE = kegg_anno [c ('pathway_id', 'gene_id')], 
                      TERM2NAME = kegg_anno [c ('pathway_id', 'pathway_description')], 
                      pvalueCutoff = 1, 
                      pAdjustMethod = 'BH', 
                      qvalueCutoff = 1, 
                      maxGSSize = 500)

# # 输出默认结果，即根据上述 p 值等阈值筛选后的
# write.table (kegg_rich, '/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/CAFE_output/node12significant.expand.KEGG.xls', sep = '\t', row.names = FALSE, quote = FALSE)
kegg_rich_res <- as.data.frame(kegg_rich)
write.table(kegg_rich_res, '/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/nodeCommon_12_15_significant.intersect.KEGG.xls', sep = '\t', row.names = FALSE, quote = FALSE)

### 4.3 绘图

> 需要注意生成的文件中有一些 term 可能是重复的，比如 GO 富集结果的 16,17 行，仔细看 geneID 那一列就会发现两行的基因是完全一样的，需要删掉一个。还有一点就是在 GO/KEGG 富集的结果中，有一些是与所研究对象物种无关的 ，这些也请酌情考虑

In [ ]:
# 读入原始GO富集文件
df <- read.delim("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/nodeCommon_12_15_significant.intersect.GO.xls", header = TRUE, sep = "\t", stringsAsFactors = FALSE)

# 去除geneID完全重复的行
# df_processed <- df[!duplicated(df$geneID), ]

# 去除Description重复，保留第一个（即p值最小的那个）
df_processed <- df[!duplicated(df$Description), ]

# 保存结果
write.table(df_processed, "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/nodeCommon_12_15_significant.intersect.GO.processed.xls", sep = "\t", row.names = FALSE, quote = FALSE)

In [ ]:
library(ggplot2)
library(dplyr)
library(ggrepel)

# 读入数据
pathway <- read.delim(
  "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/nodeCommon_12_15_significant.intersect.GO.processed.xls",
  header = TRUE, sep = "\t"
)

# GeneRatio直接用Count列
# pathway$GeneRatio <- pathway$Count / 9138
pathway$BgN <- as.numeric(sub(".*/", "", pathway$GeneRatio))   # 取出 GeneRatio 分母 (9138)
pathway$GeneRatio <- pathway$Count / pathway$BgN

# logP
pathway$logP <- -log10(pathway$pvalue)

# 生成唯一GO标签，防止Description重复造成y轴合并
pathway$GO_label <- paste0(pathway$Description, " | ", pathway$ID)

# 按pvalue或p.adjust排序，选前20
GO <- arrange(pathway, pvalue)   # 或者 p.adjust
GO_dataset <- GO[1:10, ]
GO_dataset$GO_label <- factor(GO_dataset$GO_label, levels = rev(GO_dataset$GO_label))

# Ontology如无可区分，统一赋值
GO_dataset$Ontology <- "BP"

# 绘图主题
mytheme <- theme(
  axis.title = element_text(face="bold", size=8, colour='black'),
  axis.text.y = element_text(face="bold", size=6, colour='black'),
  axis.text.x = element_text(face="bold", color="black", angle=0, vjust=1, size=8),
  axis.line = element_line(linewidth=0.5, colour = 'black'),
  panel.background = element_rect(color='black'),
  plot.title = element_text(face="bold", size=8, colour = 'black', hjust = 0.8),
  legend.key = element_blank()
)

# 气泡图（y轴唯一标识）
p <- ggplot(GO_dataset, aes(x=GeneRatio, y=GO_label, colour=logP, size=Count, shape=Ontology)) +
  geom_point() +
  scale_size(range = c(2, 8)) +
  scale_colour_gradient(low = "#52c2eb", high = "#EA4F30") +
  theme_bw() +
  labs(
    x='GeneRatio',
    y='GO Terms (Description | ID)',
    title='Enriched GO Terms',
    color=expression(-log[10](pvalue))
  ) +
  theme(
    legend.title = element_text(size=8),
    legend.text = element_text(size=14),
    axis.title.y = element_text(margin = margin(r = 50)),
    axis.title.x = element_text(margin = margin(t = 20)),
    axis.text.x = element_text(face="bold", color="black", angle=0, vjust=1)
  ) +
  mytheme

print(p)

# 保存图片
ggsave(p, filename = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/nodeCommon_12_15GO.pdf", width = 210, height = 210, units = "mm", dpi=300)
ggsave(p, filename = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/nodeCommon_12_15GO.png", width = 210, height = 210, units = "mm", dpi=300)
